# TP3. Polynômes et fonctions

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp3-polynomes-fonctions/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : évaluation naïve

Écrivez `evaluer_naif(P, x)` qui calcule $P(x)$ en suivant la définition, avec une
puissance par terme.

**Résultat attendu :** `evaluer_naif([2, 3, 0, 5], 2)` vaut `48`, car
$2 + 6 + 40 = 48$.


In [1]:
def evaluer_naif(P, x):
    total = 0
    for i in range(len(P)):
        total = total + P[i] * x ** i
    return total

print(evaluer_naif([2, 3, 0, 5], 2))

48


## Exercice 2.1 : Horner

Écrivez `evaluer(P, x)` par le schéma de Horner, et vérifiez qu'elle donne le même
résultat que la version naïve sur plusieurs points.

**Résultat attendu :** les deux fonctions coïncident, et Horner est plus rapide sur un
polynôme de degré élevé.


In [2]:
def evaluer(P, x):
    resultat = 0
    for coef in reversed(P):
        resultat = resultat * x + coef
    return resultat

P = [2, 3, 0, 5]
for x in [0, 1, 2, 3.5, -1]:
    print(x, evaluer_naif(P, x), evaluer(P, x))

0 2 2
1 10 10
2 48 48
3.5 226.875 226.875
-1 -6 -6


## Exercice 2.2 : mesurer, et se méfier de ses prévisions

Construisez des polynômes dont tous les coefficients valent 1, de degrés 1 000, 5 000
et 20 000, et comparez le temps des deux méthodes en `x = 1.0001`. Avant de lancer la
mesure, écrivez votre prévision : de combien Horner devrait-il être plus rapide ?

**Résultat attendu :** un rapport d'environ 2, **constant** quand le degré augmente.
Ce n'est probablement pas ce que vous aviez prévu.


In [3]:
import time

for degre in [1000, 5000, 20000]:
    P = [1] * (degre + 1)
    debut = time.perf_counter()
    for essai in range(10):
        evaluer_naif(P, 1.0001)
    t_naif = time.perf_counter() - debut
    debut = time.perf_counter()
    for essai in range(10):
        evaluer(P, 1.0001)
    t_horner = time.perf_counter() - debut
    print(degre, round(t_naif, 4), round(t_horner, 4), round(t_naif / t_horner, 2))

1000 0.0014 0.0005 2.61
5000 0.0069 0.0027 2.59
20000 0.0277 0.0091 3.06


## Exercice 2.3 : le cas où l'écart explose vraiment

Refaites la mesure en évaluant en `x = 3`, un **entier**, sur des degrés 200, 400
et 800. Le rapport reste-t-il constant ?

**Résultat attendu :** cette fois le rapport grandit avec le degré, d'environ 4 à
environ 6.


In [4]:
for degre in [200, 400, 800]:
    P = [1] * (degre + 1)
    debut = time.perf_counter()
    for essai in range(10):
        evaluer_naif(P, 3)
    t_naif = time.perf_counter() - debut
    debut = time.perf_counter()
    for essai in range(10):
        evaluer(P, 3)
    t_horner = time.perf_counter() - debut
    print(degre, round(t_naif, 4), round(t_horner, 4), round(t_naif / t_horner, 2))

200 0.0005 0.0001 4.14
400 0.0013 0.0002 5.3
800 0.0039 0.0006 6.53


## Exercice 3.1 : la dérivée

Écrivez `deriver(P)` qui renvoie la liste des coefficients du polynôme dérivé.

**Résultat attendu :** `deriver([2, 3, 0, 5])` vaut `[3, 0, 15]`, c'est-à-dire
$3 + 15x^{2}$.


In [5]:
def deriver(P):
    resultat = []
    for i in range(1, len(P)):
        resultat.append(i * P[i])
    return resultat

print(deriver([2, 3, 0, 5]))

[3, 0, 15]


## Exercice 4.1 : dichotomie

Écrivez `dichotomie(f, a, b, tolerance)` qui renvoie une racine de `f` entre `a` et
`b`. Appliquez-la à $x^{2} - 2$ sur $[0, 2]$.

**Résultat attendu :** une valeur proche de $\sqrt{2} \approx 1{,}414214$.


In [6]:
def dichotomie(f, a, b, tolerance):
    while b - a > tolerance:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
    return (a + b) / 2

def f(x):
    return x * x - 2

print(dichotomie(f, 0, 2, 1e-10))

1.414213562355144


## Exercice 4.2 : Newton sur un polynôme

Écrivez `newton(P, depart, tolerance)` qui cherche une racine du polynôme `P` en
utilisant `evaluer` et `deriver`. Appliquez-la à $x^{3} - 2x - 5$, en partant de 2.

**Résultat attendu :** environ `2.0945515`, atteint en cinq tours ou moins.


In [7]:
def newton(P, depart, tolerance):
    dP = deriver(P)
    x = depart
    tours = 0
    while tours < 100:
        nouveau = x - evaluer(P, x) / evaluer(dP, x)
        tours = tours + 1
        if abs(nouveau - x) < tolerance:
            return nouveau, tours
        x = nouveau
    return x, tours

racine, tours = newton([-5, -2, 0, 1], 2, 1e-10)
print(racine, tours)

2.0945514815423265 5


## Exercice 4.3 : faire échouer Newton

Appliquez `newton` au polynôme $x^{3} - 2x + 2$, c'est-à-dire `[2, -2, 0, 1]`, en
partant de `0`. Affichez le trajet des premiers itérés pour comprendre ce qui se passe.

**Résultat attendu :** aucune convergence, et un trajet qui alterne entre `0` et `1`
indéfiniment.


In [8]:
x = 0
P = [2, -2, 0, 1]
dP = deriver(P)
print(x)
for i in range(8):
    x = x - evaluer(P, x) / evaluer(dP, x)
    print(x)

0
1.0
0.0
1.0
0.0
1.0
0.0
1.0
0.0


## Exercice 4.4 : le prix d'un mauvais point de départ

Reprenez $x^{3} - 2x - 5$ et comparez le nombre de tours de `newton` en partant
de `2`, puis de `0`.

**Résultat attendu :** cinq tours depuis `2`, une vingtaine depuis `0`.


In [9]:
racine, tours = newton([-5, -2, 0, 1], 2, 1e-10)
print(tours)
racine, tours = newton([-5, -2, 0, 1], 0, 1e-10)
print(tours)

5
20


## Exercice 5.1 : lire le graphique

Sur le tracé, repérez la racine trouvée par Newton. Combien de fois la courbe
coupe-t-elle l'axe des abscisses ? Ce polynôme est de degré 3 : que dire de ses autres
racines ?


In [10]:
import matplotlib.pyplot as plt

P = [-5, -2, 0, 1]
xs = []
ys = []
for i in range(-300, 301):
    x = i / 100
    xs.append(x)
    ys.append(evaluer(P, x))

racine, tours = newton(P, 2, 1e-10)
plt.plot(xs, ys)
plt.axhline(0, color="black")
plt.plot([racine], [0], "ro")
plt.show()

print(racine)
print("La courbe coupe l'axe une seule fois. Les deux autres racines ne sont pas réelles.")

2.0945514815423265
La courbe coupe l'axe une seule fois. Les deux autres racines ne sont pas réelles.


## Exercice 6.1 : somme et produit

Écrivez `somme(P, Q)` et `produit_poly(P, Q)`. Attention, les deux listes n'ont pas
forcément la même longueur.

**Résultat attendu :** `produit_poly([1, 1], [1, 1])` vaut `[1, 2, 1]`, c'est-à-dire
$(1 + x)^{2} = 1 + 2x + x^{2}$, et `produit_poly([-1, 1], [1, 1])` vaut
`[-1, 0, 1]`, l'identité $(x - 1)(x + 1) = x^{2} - 1$.


In [11]:
def somme(P, Q):
    n = max(len(P), len(Q))
    resultat = []
    for i in range(n):
        a = 0
        b = 0
        if i < len(P):
            a = P[i]
        if i < len(Q):
            b = Q[i]
        resultat.append(a + b)
    return resultat

def produit_poly(P, Q):
    resultat = [0] * (len(P) + len(Q) - 1)
    for i in range(len(P)):
        for j in range(len(Q)):
            resultat[i + j] = resultat[i + j] + P[i] * Q[j]
    return resultat

print(somme([1, 2, 3], [1, 1]))
print(produit_poly([1, 1], [1, 1]))
print(produit_poly([-1, 1], [1, 1]))

[2, 3, 3]
[1, 2, 1]
[-1, 0, 1]


## Exercice 6.2 : le triangle de Pascal, par les polynômes

Calculez $(1 + x)^{10}$ en multipliant dix fois `[1, 1]` par lui-même. Que
reconnaissez-vous dans la liste obtenue ?

**Résultat attendu :** `[1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1]`.


In [12]:
R = [1]
for i in range(10):
    R = produit_poly(R, [1, 1])
print(R)
print("Ce sont les coefficients binomiaux, la 11e ligne du triangle de Pascal.")

[1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1]
Ce sont les coefficients binomiaux, la 11e ligne du triangle de Pascal.


## Exercice 7.1 : trouver le meilleur pas

Écrivez `derivee_avant(f, x, h)` et `derivee_centree(f, x, h)`. Calculez l'erreur de
chacune sur $\sin$ en $x = 1$, dont la dérivée exacte est $\cos 1$, pour
`h` valant $10^{-1}, 10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}, 10^{-10}, 10^{-12}$.

**Résultat attendu :** l'erreur diminue puis **remonte**. Le meilleur pas est de
l'ordre de $10^{-8}$ pour la formule avant, $10^{-6}$ pour la formule centrée,
et à $h = 10^{-12}$ l'erreur est remontée de quatre à six ordres de grandeur
par rapport à ce meilleur pas.


In [13]:
import math

def derivee_avant(f, x, h):
    return (f(x + h) - f(x)) / h

def derivee_centree(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)

exacte = math.cos(1)
for h in [1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    erreur_avant = abs(derivee_avant(math.sin, 1, h) - exacte)
    erreur_centree = abs(derivee_centree(math.sin, 1, h) - exacte)
    print(h, erreur_avant, erreur_centree)

0.1 0.04293855333275065 0.0009000536983797547
0.01 0.00421632485627077 9.004993406280803e-06
0.0001 4.207444951864758e-05 9.004295087322589e-10
1e-06 4.2074680939307996e-07 2.7716939854371958e-11
1e-08 2.9698852266335507e-09 2.581229896492232e-09
1e-10 5.848103645789138e-08 5.848103645789138e-08
1e-12 4.3240216923923214e-05 1.2270934307334613e-05


## Exercice 8.1 : les trois méthodes

Écrivez `rectangles(f, a, b, n)`, `trapezes(f, a, b, n)` et `simpson(f, a, b, n)`.
Testez-les sur $\int_0^1 x^{2}\,dx = 1/3$ pour `n` valant 10, 100 et 1000.

**Résultat attendu :** l'erreur des rectangles est divisée par 10 quand `n` est
multiplié par 10, celle des trapèzes par 100, et Simpson donne le résultat exact, à
un arrondi de $10^{-17}$ près.


In [14]:
def rectangles(f, a, b, n):
    h = (b - a) / n
    total = 0
    for i in range(n):
        total = total + f(a + i * h)
    return total * h

def trapezes(f, a, b, n):
    h = (b - a) / n
    total = (f(a) + f(b)) / 2
    for i in range(1, n):
        total = total + f(a + i * h)
    return total * h

def simpson(f, a, b, n):
    h = (b - a) / n
    total = f(a) + f(b)
    for i in range(1, n):
        if i % 2 == 1:
            total = total + 4 * f(a + i * h)
        else:
            total = total + 2 * f(a + i * h)
    return total * h / 3

def carre(x):
    return x * x

for n in [10, 100, 1000]:
    print(n, abs(rectangles(carre, 0, 1, n) - 1 / 3), abs(trapezes(carre, 0, 1, n) - 1 / 3), abs(simpson(carre, 0, 1, n) - 1 / 3))

10 0.04833333333333323 0.0016666666666667607 0.0
100 0.004983333333333284 1.6666666666720342e-05 1.1102230246251565e-16
1000 0.000499833333333366 1.6666666663445184e-07 1.6653345369377348e-16


## Exercice 8.2 : calculer π

On a $\int_0^1 \frac{4}{1 + x^{2}}\,dx = \pi$. Calculez cette intégrale par les
trapèzes et par Simpson avec `n = 1000`, et comparez à `math.pi`.

**Résultat attendu :** une erreur de l'ordre de $10^{-7}$ pour les trapèzes, et
un résultat indistinguable de `math.pi` pour Simpson.


In [15]:
def g(x):
    return 4 / (1 + x * x)

t = trapezes(g, 0, 1, 1000)
s = simpson(g, 0, 1, 1000)
print(t, abs(t - math.pi))
print(s, abs(s - math.pi))
print(math.pi)

3.1415924869231246 1.6666666846631983e-07
3.1415926535897913 1.7763568394002505e-15
3.141592653589793
